# Topological Resilience: Disease as K_nm Damage

**SCPN Claim**: The coupling topology (K_nm) provides error protection
via persistent homology. p_h1 = 0.72 on the full 16-layer complex.
Neurodegenerative diseases damage specific layers, degrading p_h1.

**Novel prediction**: The RATE of p_h1 decline under targeted damage
should predict disease progression timelines. Diseases that damage
high-connectivity layers (high K row-sum) should cause faster p_h1
collapse than diseases targeting peripheral layers.

**Diseases modelled**:
- Alzheimer's: L9 (memory) + L5 (emotional) damage
- Parkinson's: L2 (neurochemical dopamine) damage
- Epilepsy: L4-L5 hypercoupling (too much K, not too little)
- ALS: L4 (cellular motor neuron) damage
- Schizophrenia: L7 (symbolic binding) + L10 (boundary) disruption

In [ ]:
import numpy as np

from scpn_quantum_control.bridge import build_knm_paper27

In [ ]:
def coupling_weighted_persistent_h1(K, max_weight=None):
    """Compute p_h1 on the coupling-weighted simplicial complex.

    Filtration: add edges in DECREASING K order (strongest first).
    H1 features born when a cycle forms in the strong-coupling subgraph.
    H1 features die when a triangle (2-simplex) fills the cycle.

    p_h1 = (total H1 persistence) / (max possible persistence)
    """
    N = K.shape[0]
    if max_weight is None:
        max_weight = np.max(K)

    # Build edge list sorted by DECREASING coupling (strongest first)
    edges = []
    for i in range(N):
        for j in range(i + 1, N):
            if K[i, j] > 1e-10:
                edges.append((K[i, j], i, j))
    edges.sort(reverse=True)  # strongest coupling first

    # Track connected components and cycles
    parent = list(range(N))
    rank = [0] * N

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx == ry:
            return False
        if rank[rx] < rank[ry]:
            rx, ry = ry, rx
        parent[ry] = rx
        if rank[rx] == rank[ry]:
            rank[rx] += 1
        return True

    h1_births = []  # coupling values where cycles form
    adjacency = set()

    for weight, i, j in edges:
        if union(i, j):
            adjacency.add((i, j))
        else:
            h1_births.append(weight)
            adjacency.add((i, j))

    # Approximate H1 deaths: cycle killed when triangle forms
    # For each H1 birth, find the weakest edge needed to form a triangle
    h1_deaths = []
    for birth_weight in h1_births:
        # Rough: death when next weaker edge completes a triangle
        death = birth_weight * 0.5  # approximate
        h1_deaths.append(death)

    # Persistence
    total_persistence = sum(b - d for b, d in zip(h1_births, h1_deaths))
    max_persistence = max_weight * len(h1_births) if h1_births else 1.0
    p_h1 = total_persistence / max_persistence if max_persistence > 0 else 0.0

    return {
        "p_h1": p_h1,
        "n_cycles": len(h1_births),
        "total_persistence": total_persistence,
        "h1_births": h1_births,
        "h1_deaths": h1_deaths,
    }


# Baseline: healthy K_nm
K16 = build_knm_paper27(L=16)
baseline = coupling_weighted_persistent_h1(K16)
print("Healthy K_nm (16x16):")
print(f"  p_h1 = {baseline['p_h1']:.4f}")
print(f"  H1 cycles = {baseline['n_cycles']}")
print(f"  Total persistence = {baseline['total_persistence']:.4f}")

In [ ]:
# Disease models: targeted K_nm damage
DISEASES = {
    "Alzheimer's": {
        "layers": [9, 5],  # L9 memory + L5 emotional
        "mechanism": "tau tangles destroy hippocampal + entorhinal coupling",
        "clinical_years": 8,  # mean years from diagnosis to severe
    },
    "Parkinson's": {
        "layers": [2],  # L2 neurochemical (dopamine)
        "mechanism": "substantia nigra dopamine neuron death",
        "clinical_years": 15,
    },
    "ALS": {
        "layers": [4],  # L4 cellular (motor neuron)
        "mechanism": "upper + lower motor neuron degeneration",
        "clinical_years": 3,
    },
    "Schizophrenia": {
        "layers": [7, 10],  # L7 symbolic + L10 boundary
        "mechanism": "gamma binding deficit + self-other boundary dissolution",
        "clinical_years": 30,  # chronic, slow progression
    },
    "Epilepsy": {
        "layers": [4, 5],  # L4-L5 HYPERcoupling
        "mechanism": "GABAergic failure -> excitatory dominance -> hypersync",
        "clinical_years": 20,
        "mode": "hyper",  # increase K instead of decrease
    },
}


def damage_knm(K, target_layers, fraction, mode="hypo"):
    """Damage K_nm by reducing (hypo) or increasing (hyper) target layer couplings."""
    K_damaged = K.copy()
    for layer in target_layers:
        idx = layer - 1  # 0-indexed
        if mode == "hypo":
            K_damaged[idx, :] *= 1 - fraction
            K_damaged[:, idx] *= 1 - fraction
        elif mode == "hyper":
            K_damaged[idx, :] *= 1 + fraction * 3  # 3x amplification
            K_damaged[:, idx] *= 1 + fraction * 3
    # Keep symmetric
    K_damaged = (K_damaged + K_damaged.T) / 2
    return K_damaged


# Simulate progressive damage for each disease
damage_fractions = np.linspace(0, 0.95, 20)

print(
    f"\n{'Disease':<18} {'Layers':<10} {'p_h1@0%':>8} {'p_h1@50%':>9} {'p_h1@95%':>9} {'Collapse@':>10} {'Clinical':>9}"
)
print("-" * 82)

disease_results = {}
for name, params in DISEASES.items():
    mode = params.get("mode", "hypo")
    p_h1_curve = []
    for frac in damage_fractions:
        K_d = damage_knm(K16, params["layers"], frac, mode)
        result = coupling_weighted_persistent_h1(K_d)
        p_h1_curve.append(result["p_h1"])

    # Find collapse point (p_h1 drops below 50% of baseline)
    collapse_frac = None
    for i, p in enumerate(p_h1_curve):
        if p < baseline["p_h1"] * 0.5:
            collapse_frac = damage_fractions[i]
            break

    layers_str = ",".join(f"L{l}" for l in params["layers"])
    p50_idx = len(damage_fractions) // 2

    print(
        f"{name:<18} {layers_str:<10} {p_h1_curve[0]:8.4f} {p_h1_curve[p50_idx]:9.4f} {p_h1_curve[-1]:9.4f} {collapse_frac if collapse_frac else 'never':>10} {params['clinical_years']:>7}yr"
    )

    disease_results[name] = {
        "layers": params["layers"],
        "p_h1_curve": p_h1_curve,
        "collapse_fraction": collapse_frac,
        "clinical_years": params["clinical_years"],
    }

In [ ]:
# KEY TEST: Does collapse rate correlate with clinical progression speed?
# Faster clinical progression -> faster p_h1 collapse

clinical_years = []
collapse_fracs = []
names = []

for name, res in disease_results.items():
    if res["collapse_fraction"] is not None:
        clinical_years.append(res["clinical_years"])
        collapse_fracs.append(res["collapse_fraction"])
        names.append(name)

print("\n=== TOPOLOGICAL RESILIENCE vs CLINICAL PROGRESSION ===")
print()

if len(clinical_years) >= 3:
    from scipy.stats import pearsonr, spearmanr

    r_p, p_p = pearsonr(collapse_fracs, clinical_years)
    r_s, p_s = spearmanr(collapse_fracs, clinical_years)

    print(f"Pearson:  r={r_p:.4f}, p={p_p:.4f}")
    print(f"Spearman: r={r_s:.4f}, p={p_s:.4f}")
    print()

    for n, cf, cy in zip(names, collapse_fracs, clinical_years):
        print(f"  {n:<18}: collapse@{cf:.0%} damage, clinical {cy}yr")

    print()
    if r_p > 0.5 and p_p < 0.1:
        print("POSITIVE CORRELATION: more damage needed to collapse = slower disease.")
        print("The K_nm topology PREDICTS clinical progression rate.")
        print("This is a NOVEL testable prediction of the SCPN.")
    elif r_p < -0.5:
        print("NEGATIVE CORRELATION: easier to collapse = slower disease.")
        print("Unexpected. May indicate compensatory mechanisms.")
    else:
        print("NO SIGNIFICANT CORRELATION.")
        print("Topological resilience does not predict clinical timeline.")
        print("Caveats: only 5 diseases, progression is multifactorial.")
else:
    print(f"Only {len(clinical_years)} diseases with defined collapse points.")
    print("Need more data points for correlation.")

In [ ]:
# Layer vulnerability analysis
print("\n=== LAYER VULNERABILITY RANKING ===")
print("Which layers, when damaged, cause the fastest p_h1 collapse?\n")

print(
    f"{'Layer':<8} {'K row sum':>10} {'p_h1@50%dmg':>13} {'p_h1@95%dmg':>13} {'Vulnerability':>14}"
)
print("-" * 62)

vulnerabilities = []
for layer in range(1, 17):
    K_d50 = damage_knm(K16, [layer], 0.5)
    K_d95 = damage_knm(K16, [layer], 0.95)
    p50 = coupling_weighted_persistent_h1(K_d50)["p_h1"]
    p95 = coupling_weighted_persistent_h1(K_d95)["p_h1"]
    row_sum = np.sum(K16[layer - 1, :]) - K16[layer - 1, layer - 1]
    vuln = baseline["p_h1"] - p95  # larger = more vulnerable
    vulnerabilities.append((layer, row_sum, p50, p95, vuln))

vulnerabilities.sort(key=lambda x: -x[4])  # most vulnerable first
for layer, rs, p50, p95, vuln in vulnerabilities:
    print(f"L{layer:<7} {rs:10.3f} {p50:13.4f} {p95:13.4f} {vuln:14.4f}")

print()
print("PREDICTION: Diseases affecting the TOP vulnerability layers")
print("should have the most devastating clinical impact.")
most_vuln = vulnerabilities[0]
least_vuln = vulnerabilities[-1]
print(f"Most vulnerable:  L{most_vuln[0]} (K row sum = {most_vuln[1]:.3f})")
print(f"Least vulnerable: L{least_vuln[0]} (K row sum = {least_vuln[1]:.3f})")

In [ ]:
import json

print("--- JSON RESULTS ---")
out = {
    "baseline_p_h1": round(baseline["p_h1"], 4),
    "baseline_n_cycles": baseline["n_cycles"],
    "diseases": {},
}
for name, res in disease_results.items():
    out["diseases"][name] = {
        "layers": res["layers"],
        "collapse_fraction": res["collapse_fraction"],
        "clinical_years": res["clinical_years"],
        "p_h1_at_50pct": round(res["p_h1_curve"][len(damage_fractions) // 2], 4),
        "p_h1_at_95pct": round(res["p_h1_curve"][-1], 4),
    }
if len(clinical_years) >= 3:
    out["correlation_pearson_r"] = round(r_p, 4)
    out["correlation_pearson_p"] = round(p_p, 4)
out["most_vulnerable_layer"] = vulnerabilities[0][0]
out["least_vulnerable_layer"] = vulnerabilities[-1][0]
print(json.dumps(out, indent=2))